In [1]:
from data_classes.sample_noise_profile import SampleNoiseProfile
import numpy as np
import re
from typing import List
import importlib
import warnings
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# from LNCRDNCN import NoiseClassifier
import LNCRDNCN

NoiseClassifier = LNCRDNCN.NoiseClassifier

importlib.reload(LNCRDNCN) # for module development with notebook
warnings.filterwarnings('ignore')

In [2]:
X = np.array([
    [1.5, 1.5], [2.0, 2.2], [2.5, 1.8], [1.8, 2.5], # Class 0 cluster
    [7.5, 7.5], [6.8, 7.2], [7.2, 6.8], [8.0, 7.0], # Class 1 cluster
    
    [2.1, 2.1], # Index 8: Target Clean Sample (Labeled 0, physically inside Class 0)
    [2.0, 2.1]  # Index 9: Target Noisy Sample (Labeled 1, but physically inside Class 0!)
])

y_clean = None
y_noisy = np.array(["0", "0", '0', '0', '1', '1', '0', '1', '0', '1'])
# y_clean = np.array([0, 0, 0, 0, 1, 0, 1, 1, 0, 0])

n_samples = len(X)

# y = y[np.newaxis,:]

n_classes = len(np.unique(y_noisy))
TESTING = False

# print(y_noisy)


In [3]:
def build_noise_profiles(
    n_samples,
    iteration_records,
    active_mask,
    adaptive_remove_mask,
    y_working,
    y_noisy,
    y_clean_reference,
    is_noisy_ground_truth,
    testing = TESTING
):
    sample_best_rns = {}       # sample_idx → highest rNS observed
    sample_final_decision = {} # sample_idx → last decision
    sample_confidence = {}     # sample_idx → confidence at detection
    sample_ever_in_NF = set()

    for record in iteration_records:
        for dec in record.decisions:
            idx = dec.sample_idx
            sample_ever_in_NF.add(idx)
            # Keep track of the maximum rNS observed (most noisy iteration)
            if idx not in sample_best_rns or dec.rns_score > sample_best_rns[idx]:
                sample_best_rns[idx] = dec.rns_score
            # Update final decision (last iteration wins)
            sample_final_decision[idx] = dec.decision
            sample_confidence[idx] = dec.confidence

    # Collect all rNS̄ values (for final threshold)
    if iteration_records:
        final_rns_mean = iteration_records[-1].rns_mean
    else:
        final_rns_mean = 0.0

    # Build profiles
    raw_profiles = []
    for i in range(n_samples):
        in_NF = i in sample_ever_in_NF
        rns = sample_best_rns.get(i, float('-inf'))
        conf = sample_confidence.get(i, 1.0)
        final_dec = sample_final_decision.get(i, "NONE")

        # Determine if adaptively removed
        adaptively_removed = bool(adaptive_remove_mask[i])
        truly_deleted = (not active_mask[i]) or adaptively_removed

        # --- Noise Category (our extension for interpretability) ---
        if not in_NF:
            category = "Clean"
            action = "Keep"
        elif rns < 0:
            category = "Suspicious"
            action = "Keep"
        elif rns < final_rns_mean:
            category = "High Noise"
            action = "Correct/Fix"
        else:
            category = "Critical Noise"
            if truly_deleted:
                action = "Remove"
            else:
                action = "Correct/Fix"

        raw_profiles.append({
            'idx': i,
            'rns': rns,
            'conf': conf,
            'in_NF': in_NF,
            'final_dec': final_dec,
            'category': category,
            'action': action,
            'true_label': y_clean_reference[i] if testing else None,
            'working_label': y_working[i],
            'source_label': y_noisy[i],
            'is_truly_noisy': bool(is_noisy_ground_truth[i]) if testing else None
        })

    # Sort by noise score descending (most noisy first); NaN-like -inf goes last
    raw_profiles_sorted = sorted(
        raw_profiles,
        key=lambda x: x['rns'],
        reverse=True
    )

    profiles = []
    for rank, p in enumerate(raw_profiles_sorted, start=1):
        profiles.append(SampleNoiseProfile(
            sample_idx=p['idx'],
            noise_score=p['rns'],
            noise_rank=rank,
            noise_category=p['category'],
            simulated_action=p['action'],
            confidence=p['conf'],
            was_ever_in_NF=p['in_NF'],
            final_decision=p['final_dec'],
            true_label=p['true_label'],
            source_label=p['source_label'],
            working_label=p['working_label'],
            is_truly_noisy=p['is_truly_noisy']
        ))

    return profiles


In [4]:
def compute_detection_metrics(
    noise_profiles: List[SampleNoiseProfile],
    testing = TESTING
):
    y_true_binary = np.array([1 if p.is_truly_noisy else 0 for p in noise_profiles])
    y_pred_binary = np.array([1 if p.was_ever_in_NF else 0 for p in noise_profiles])

    acc = accuracy_score(y_true_binary, y_pred_binary)
    prec = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    rec = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    cm = confusion_matrix(y_true_binary, y_pred_binary)

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'confusion_matrix': cm,
        'y_true': y_true_binary,
        'y_pred': y_pred_binary,
        'n_true_noisy': y_true_binary.sum(),
        'n_detected_noisy': y_pred_binary.sum(),
    }


def compute_decision_quality_metrics(
    noise_profiles: List[SampleNoiseProfile],
    n_samples: int
):
    metrics = {
        'Well Corrected': 0,
        'Well Filtered': 0,
        'Well Unfiltered': 0,
        'Wrongly Corrected': 0,
        'Wrongly Filtered': 0,
        'Wrongly Unfiltered': 0,
    }

    for p in noise_profiles:
        truly_noisy = p.is_truly_noisy
        action = p.simulated_action
        dec = p.final_decision

        # Well corrected: truly noisy → correctly relabelled
        if truly_noisy and dec == 'CORRECT' and p.working_label == p.true_label:
            metrics['Well Corrected'] += 1

        # Well filtered: truly noisy → deleted
        elif truly_noisy and action == 'Remove':
            metrics['Well Filtered'] += 1

        # Wrongly corrected: clean → relabelled (incorrectly)
        elif not truly_noisy and dec == 'CORRECT':
            metrics['Wrongly Corrected'] += 1

        # Wrongly filtered: clean → deleted
        elif not truly_noisy and action == 'Remove':
            metrics['Wrongly Filtered'] += 1

        # Wrongly unfiltered: noisy → kept unchanged
        elif truly_noisy and action == 'Keep' and dec not in ('CORRECT', 'DELETE'):
            metrics['Wrongly Unfiltered'] += 1

        # Well unfiltered: clean → correctly kept
        elif not truly_noisy and action == 'Keep':
            metrics['Well Unfiltered'] += 1

    # Convert to percentages
    df = pd.DataFrame([
        {
            'Metric': k,
            'Count': v,
            'Percentage (%)': round(100 * v / n_samples, 3),
            'Category': 'Good Decision' if k.startswith('Well') else 'Bad Decision'
        }
        for k, v in metrics.items()
    ])
    return df


def create_noise_profile_dataframe(noise_profiles: List[SampleNoiseProfile]) -> pd.DataFrame:
    rows = []
    for p in noise_profiles:
        rows.append({
            'Sample Index': p.sample_idx,
            'Noise Rank': p.noise_rank,
            'Noise Score': round(p.noise_score, 6) if p.noise_score != float('-inf') else None,
            'Noise Category': p.noise_category,
            'Simulated Action': p.simulated_action,
            'Confidence': round(p.confidence, 4),
            'Was in NF': p.was_ever_in_NF,
            'Final Decision': p.final_decision,
            'True Label': p.true_label,
            'Source Label': p.source_label,
            'Working Label': p.working_label,
            'Is Truly Noisy': p.is_truly_noisy,
            'Correctly Handled': (
                # True if: noisy was detected, or clean was kept
                (p.is_truly_noisy and p.was_ever_in_NF) or
                (not p.is_truly_noisy and not p.was_ever_in_NF)
            )
        })
    return pd.DataFrame(rows)


# --- Run Evaluation ---
def eval(n_samples,noise_profiles,iteration_records,active_mask,adaptive_remove_mask,testing=TESTING):
    profile_df = create_noise_profile_dataframe(noise_profiles)
    detection_metrics = compute_detection_metrics(noise_profiles)
    decision_quality_df = compute_decision_quality_metrics(noise_profiles, n_samples)

    # --- Print Noise Detection Statistics ---
    print("\n--- 1. NOISE DETECTION STATISTICS ---")
    print(f"  Detected as noisy (ever in NF) : {detection_metrics['n_detected_noisy']}")
    print(f"  Total samples                  : {n_samples}")
    if TESTING:
        print(f"  True noisy samples (injected)  : {detection_metrics['n_true_noisy']}")
        print(f"  Accuracy                        : {detection_metrics['accuracy']:.4f}")
        print(f"  Precision                       : {detection_metrics['precision']:.4f}")
        print(f"  Recall                          : {detection_metrics['recall']:.4f}")
        print(f"  F1-Score                        : {detection_metrics['f1_score']:.4f}")

        print("\n--- 2. CONFUSION MATRIX (Noisy Detection) ---")
        cm = detection_metrics['confusion_matrix']
        cm_df = pd.DataFrame(
            cm,
            index=['True: Clean', 'True: Noisy'],
            columns=['Pred: Clean', 'Pred: Noisy']
        )
        print(cm_df.to_string())

    # --- Print Noise Category Distribution ---
    print("\n--- 3. NOISE CATEGORY DISTRIBUTION ---")
    cat_counts = profile_df['Noise Category'].value_counts()
    cat_pct = (cat_counts / n_samples * 100).round(2)
    cat_summary = pd.DataFrame({'Count': cat_counts, 'Percentage (%)': cat_pct})
    print(cat_summary.to_string())

    # --- Print Simulated Action Distribution ---
    print("\n--- 4. SIMULATED ACTION DISTRIBUTION ---")
    action_counts = profile_df['Simulated Action'].value_counts()
    action_pct = (action_counts / n_samples * 100).round(2)
    action_summary = pd.DataFrame({'Count': action_counts, 'Percentage (%)': action_pct})
    print(action_summary.to_string())

    # --- Print Noise Ranking Statistics ---
    print("\n--- 5. NOISE SCORE DISTRIBUTION (detected samples only) ---")
    detected_df = profile_df[profile_df['Was in NF'] == True].copy()
    if len(detected_df) > 0:
        print(detected_df['Noise Score'].describe().round(4).to_string())

    # --- Print Top-N Most Suspicious Samples ---
    N = 20
    print(f"\n--- 6. TOP-{N} MOST SUSPICIOUS SAMPLES ---")
    columns = [
        'Sample Index', 'Noise Rank', 'Noise Score', 'Noise Category',
        'Simulated Action', 'Confidence','Source Label' ,'Working Label'
    ]

    if testing:
        columns += [
            'True Label',
            'Is Truly Noisy'
        ]

    top_n = profile_df.sort_values(by='Noise Rank').head(N)[columns]
    print(top_n.to_string(index=False))

    # --- Print Iteration History ---
    print("\n--- 7. ITERATION HISTORY ---")
    iter_summary = []
    for rec in iteration_records:
        iter_summary.append({
            'Iteration': rec.iteration + 1,
            'Active Samples': rec.n_active,
            '|NF|': rec.NF_size,
            'rNS̄': round(rec.rns_mean, 4),
            'Retained': rec.n_retained,
            'Corrected': rec.n_corrected,
            'Deleted': rec.n_deleted
        })
    iter_df = pd.DataFrame(iter_summary)
    print(iter_df.to_string(index=False))

    if testing:
        # --- Print Decision Quality Metrics ---
        print("\n--- 8. DECISION QUALITY METRICS ---")
        good = decision_quality_df[decision_quality_df['Category'] == 'Good Decision']
        bad  = decision_quality_df[decision_quality_df['Category'] == 'Bad Decision']
        print("\nGood Decisions:")
        print(good[['Metric', 'Count', 'Percentage (%)']].to_string(index=False))
        print("\nBad Decisions:")
        print(bad[['Metric', 'Count', 'Percentage (%)']].to_string(index=False))

    # --- Overall Summary ---
    print("\n--- 9. FINAL SUMMARY ---")
    n_active_final = active_mask.sum()
    n_adaptive = adaptive_remove_mask.sum()
    n_total_removed = (~active_mask).sum() + n_adaptive
    n_corrected_total = sum(
        1 for p in noise_profiles
        if p.final_decision == 'CORRECT'
    )
    print(f"  Original dataset size     : {n_samples}")
    print(f"  Samples after main loop   : {active_mask.sum()}")
    print(f"  Adaptive removals         : {n_adaptive}")
    print(f"  Total effectively removed : {n_total_removed}")
    print(f"  Total corrected (simul.)  : {n_corrected_total}")
    print(f"  Final active dataset size : {max(0, n_active_final - n_adaptive)}")
    print(f"  Noise detected rate      : {(detection_metrics['n_detected_noisy']/n_samples):.1%}")
    print(f"  Iterations completed      : {len(iteration_records)}")

    return detection_metrics, detected_df, profile_df

In [5]:
# Style settings
plt.style.use('seaborn-v0_8-darkgrid')
FIGSIZE_STD = (10, 5)
CMAP = 'viridis'
COLOR_PALETTE = {
    'Clean': '#2ecc71',
    'Suspicious': '#f39c12',
    'High Noise': '#e74c3c',
    'Critical Noise': '#8e44ad'
}

# ---- Chart 1: Noise Score Distribution ----
def charts(detection_metrics, detected_df, profile_df,iteration_records):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('LNC-RDNCN: Noise Detection Analysis — Wine Dataset\n'
                #  f"(Noise injection rate: {(detection_metrics['n_detected_noisy']/n_samples):.0%}, "
                f'{len(iteration_records)} iterations)',
                fontsize=14, fontweight='bold')
    ax1 = axes[0, 0]
    detected_scores = detected_df['Noise Score'].dropna()
    if len(detected_scores) > 0:
        ax1.hist(detected_scores, bins=20, edgecolor='black',
                color='steelblue', alpha=0.8, label='rNS scores')
        if len(iteration_records) > 0:
            rns_mean_final = iteration_records[-1].rns_mean
            ax1.axvline(rns_mean_final, color='red', linestyle='--', linewidth=2,
                        label=f'rNS̄ = {rns_mean_final:.3f}')
        ax1.axvline(0, color='orange', linestyle=':', linewidth=2, label='rNS = 0')
        ax1.set_xlabel('Noise Score (rNS)', fontsize=11)
        ax1.set_ylabel('Count', fontsize=11)
        ax1.set_title('Noise Score Distribution\n(Detected Noisy Samples)', fontsize=11)
        ax1.legend(fontsize=9)
    else:
        ax1.text(0.5, 0.5, 'No detected noisy samples', ha='center', va='center',
                transform=ax1.transAxes)

    # ---- Chart 2: Noise Category Distribution ----
    ax2 = axes[0, 1]
    categories_ordered = ['Clean', 'Suspicious', 'High Noise', 'Critical Noise']
    cat_data = profile_df['Noise Category'].value_counts()
    cat_values = [cat_data.get(c, 0) for c in categories_ordered]
    colors = [COLOR_PALETTE[c] for c in categories_ordered]
    bars = ax2.bar(categories_ordered, cat_values, color=colors, edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, cat_values):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                str(val), ha='center', va='bottom', fontsize=10)
    ax2.set_xlabel('Noise Category', fontsize=11)
    ax2.set_ylabel('Number of Samples', fontsize=11)
    ax2.set_title('Noise Category Distribution\n(All Samples)', fontsize=11)

    # ---- Chart 3: Top Noisy Samples (Horizontal bar) ----
    ax3 = axes[0, 2]
    top_plot = profile_df.head(15).copy()
    top_plot = top_plot[top_plot['Noise Score'].notna()]
    if len(top_plot) > 0:
        colors_top = [COLOR_PALETTE.get(c, 'grey') for c in top_plot['Noise Category']]
        bars = ax3.barh(
            [f"S{idx}" for idx in top_plot['Sample Index']],
            top_plot['Noise Score'],
            color=colors_top, edgecolor='black', alpha=0.85
        )
        ax3.set_xlabel('Noise Score (rNS)', fontsize=11)
        ax3.set_ylabel('Sample ID', fontsize=11)
        ax3.set_title('Top-15 Noisiest Samples', fontsize=11)
        # Legend patches
        patches = [mpatches.Patch(color=v, label=k) for k, v in COLOR_PALETTE.items()
                if k != 'Clean']
        ax3.legend(handles=patches, fontsize=8, loc='lower right')

    if TESTING:
        # ---- Chart 4: Confusion Matrix Heatmap ----
        ax4 = axes[1, 0]
        cm = detection_metrics['confusion_matrix']
        if cm.shape == (2, 2):
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
                        xticklabels=['Pred: Clean', 'Pred: Noisy'],
                        yticklabels=['True: Clean', 'True: Noisy'],
                        linewidths=0.5, cbar=False)
            ax4.set_title(
                f'Confusion Matrix — Noise Detection\n'
                f'Accuracy={detection_metrics["accuracy"]:.3f}, '
                f'F1={detection_metrics["f1_score"]:.3f}',
                fontsize=11
            )

    # ---- Chart 5: Iteration History ----
    ax5 = axes[1, 1]
    if len(iteration_records) > 0:
        iters = [r.iteration + 1 for r in iteration_records]
        rns_means = [r.rns_mean for r in iteration_records]
        nf_sizes = [r.NF_size for r in iteration_records]

        ax5_twin = ax5.twinx()
        ax5.plot(iters, rns_means, 'b-o', label='rNS̄ (mean noise score)', linewidth=2)
        ax5_twin.bar(iters, nf_sizes, alpha=0.3, color='red', label='|NF| size')
        ax5.set_xlabel('Iteration', fontsize=11)
        ax5.set_ylabel('Mean Noise Score (rNS̄)', color='blue', fontsize=11)
        ax5_twin.set_ylabel('|NF| Size', color='red', fontsize=11)
        ax5.set_title('Iteration History:\nNoise Score & Noise Set Size', fontsize=11)
        ax5.legend(loc='upper left', fontsize=9)
        ax5_twin.legend(loc='upper right', fontsize=9)

    # ---- Chart 6: Clean Function (Fig. 4 from paper) ----
    ax6 = axes[1, 2]
    rd_values = np.linspace(0, 1, 200)

    # clean(ej) for clean sample: 1 - 0.5*Rd
    clean_for_clean = 1.0 - 0.5 * rd_values
    # clean(ej) for noisy sample: 0.5 * (1 - Rd)
    clean_for_noisy = 0.5 * (1.0 - rd_values)

    ax6.plot(rd_values, clean_for_clean, 'g-', linewidth=2.5,
            label='clean(ej) — ej is Clean sample')
    ax6.plot(rd_values, clean_for_noisy, 'r--', linewidth=2.5,
            label='clean(ej) — ej is Noisy sample')
    ax6.axhline(0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7, label='Ambiguity boundary = 0.5')
    ax6.fill_between(rd_values, 0.5, clean_for_clean, alpha=0.1, color='green')
    ax6.fill_between(rd_values, clean_for_noisy, 0.5, alpha=0.1, color='red')
    ax6.set_xlabel('Relative Noise Density Rd(NF, ej)', fontsize=11)
    ax6.set_ylabel('clean(ej)', fontsize=11)
    ax6.set_title('Clean Function (Eq. 8)\n[Reproducing Fig. 4 from Paper]', fontsize=11)
    ax6.legend(fontsize=9)
    ax6.set_ylim(-0.05, 1.05)
    ax6.set_xlim(0, 1)

    plt.tight_layout()
    plt.savefig('lnc_rdncn_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("[✓] Visualisations saved to 'lnc_rdncn_analysis.png'")


In [6]:
def parse_custom_arff(file_path):
    column_names = []
    data_started = False
    data_rows = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            # 1. Capture attribute names (ignores range descriptors like real[...])
            if line.lower().startswith('@attribute'):
                # Extract the word immediately following @attribute
                match = re.search(r'@attribute\s+([^\s{]+)', line, re.IGNORECASE)
                if match:
                    column_names.append(match.group(1))
                    
            # 2. Flag the start of data rows
            elif line.lower().startswith('@data'):
                data_started = True
                
            # 3. Collect row instances, ignoring trailing custom tags like @inputs
            elif data_started and not line.startswith('@'):
                # Split comma-separated tokens and strip whitespace
                row_values = [val.strip() for val in line.split(',')]
                data_rows.append(row_values)

    # 4. Construct the pandas DataFrame
    df = pd.DataFrame(data_rows, columns=column_names)
    
    # 5. Automatically convert numeric columns from strings to float/int
    df = df.apply(pd.to_numeric, errors='ignore')
    
    return df


In [7]:
df = pd.concat([parse_custom_arff(f'data/paper_dataset/base/autos/autos-5-{i}tra.dat') for i in range(1,2)]) 

df['num-of-cylinders'] = df['num-of-cylinders'].map({'three':3,'twelve':12,'six':6,'two':2,'four':4,'five':5,'eight':8})
df['num-of-doors'] = df['num-of-doors'].map({'four':4,'two':2})

y = df['make']
df.drop(columns=['make','normalized-losses'],inplace=True)

df = pd.get_dummies(df,dtype=int)

print(df.info())

TESTING = False
X = df.to_numpy()
y_noisy = y.to_numpy()
y_clean = None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 41 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   num-of-doors            127 non-null    int64  
 1   wheel-base              127 non-null    float64
 2   length                  127 non-null    float64
 3   width                   127 non-null    float64
 4   height                  127 non-null    float64
 5   curb-weight             127 non-null    float64
 6   num-of-cylinders        127 non-null    int64  
 7   engine-size             127 non-null    float64
 8   bore                    127 non-null    float64
 9   stroke                  127 non-null    float64
 10  compression-ratio       127 non-null    float64
 11  horsepower              127 non-null    float64
 12  peak-rpm                127 non-null    float64
 13  city-mpg                127 non-null    float64
 14  highway-mpg             127 non-null    fl

In [16]:
nc = NoiseClassifier(k_ncn=5,voting_threshold=0.5,cv_folds=10,max_iterations=4)

active_mask, y_working, iteration_records, adaptive_remove_mask = nc.fit(X,y_noisy)

noise_profiles = build_noise_profiles(
    n_samples=n_samples,
    iteration_records=iteration_records,
    active_mask=active_mask,
    adaptive_remove_mask=adaptive_remove_mask,
    y_noisy=y_noisy,
    y_working=y_working,
    y_clean_reference=y_clean if TESTING else None,
    is_noisy_ground_truth= y_clean != y_noisy if TESTING else None,
    testing = TESTING
)

ob = eval(len(X),noise_profiles,iteration_records,active_mask,adaptive_remove_mask,TESTING)


--- 1. NOISE DETECTION STATISTICS ---
  Detected as noisy (ever in NF) : 2
  Total samples                  : 127

--- 3. NOISE CATEGORY DISTRIBUTION ---
                Count  Percentage (%)
Noise Category                       
Clean               8            6.30
High Noise          2            1.57

--- 4. SIMULATED ACTION DISTRIBUTION ---
                  Count  Percentage (%)
Simulated Action                       
Keep                  8            6.30
Correct/Fix           2            1.57

--- 5. NOISE SCORE DISTRIBUTION (detected samples only) ---
count    2.0000
mean     0.1399
std      0.1534
min      0.0314
25%      0.0857
50%      0.1399
75%      0.1941
max      0.2484

--- 6. TOP-20 MOST SUSPICIOUS SAMPLES ---
 Sample Index  Noise Rank  Noise Score Noise Category Simulated Action  Confidence  Source Label Working Label
            3           1     0.248369     High Noise      Correct/Fix      0.2500 mercedes-benz mercedes-benz
            0           2     0.03144